希望通过 CoT + 本地数据库，给 Agent 更大的权限，来进行文本理解和处理；

数据库部分，拟选取 SQLite 存储，包括提取结果和原始文献两部分；

Agent 部分，需要尝试
  * CoT 结构输出，调用工具，并衔接上下文
  * 从数据库中获取需要的数据
  * 写入更新数据库
  * 无法解决的问题，悬置并标记
最终逐步将原始文献（《辞典》）转变为结构化数据，并保留全部的处理痕迹以及溯源信息；


In [2]:
"""
首先构建数据库的基本格式
需要有清晰的结构，以及足够的可扩展性
"""

"""
对于原始文献《辞典》而言，结构固定，且基本不需要修改；
结合 "第八至十编-表格化结果.json" 与 "职官条目分类目录.json" 两个文件，构建该表
  * 使用自然索引
  * title 条目名称
  * catalog 条目所在目录
  * page 条目所在页码
  * text 条目自带文本
  * fields 条目各字段内容（json2str）
"""

"""
不能直接使用条目名称作为索引，即使是在八至十编，也存在重名；
包括：宣抚大使、经制司、观察判官、节度推官、观察推官、录事参军事、司户参军事、司理参军事、司法参军事；
不同级别的地方机构可能设立相同名称的职位；
"""

import json

pwd = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\refined_tables"
with open(pwd + r"\职官条目分类目录.json", "r", encoding="utf-8") as f:
  all_catalog_data = json.load(f)
  catalog_data = []
  ff = True
  for item in all_catalog_data:
    if ff and item["text"] != "第八编 军事统率机构与地方治安机构类":
      continue
    else:
      ff = False
    if item["text"] == "第十一编 阶官类":
      break
    catalog_data.append(item)
  
  hh = {
    "catalog": 1,
    "h1": 2,
    "h2": 3,
    "h3": 4
  }
  entries_catalog_data = []
  current_hierarchy = [[0, "宋代官制辞典"], [1, "I.职官条目分类目录"]]
  for item in catalog_data:
    if item["type"] in hh:
      ind = hh[item["type"]]
      text = item["text"]
      while current_hierarchy[-1][0] >= ind:
        current_hierarchy.pop()
      current_hierarchy.append([ind, text])
    elif item["type"] == "name":
      entries_catalog_data.append({
        "text": item["text"],
        "catalog": "/".join([x[1] for x in current_hierarchy]),
        "page": item["page"]
      })

with open(pwd + r"\第八至十编-表格化结果.json", "r", encoding="utf-8") as f:
  entries_data = json.load(f)

print(len(entries_data), len(entries_catalog_data))
print(json.dumps(entries_catalog_data[0], ensure_ascii=False, indent=2))

assert len(entries_data) == len(entries_catalog_data)

entry_list = []
for cata, item in zip(entries_catalog_data, entries_data):
  assert cata["text"] == item["name"], f"Miss Match: {cata["text"]} {item["name"]}"
  fields = {}
  for key, value in item.items():
    if key not in ["name", "text"]:
      fields[key] = value
  entry_list.append({
    "title": cata["text"],
    "catalog": cata["catalog"],
    "page": cata["page"],
    "text": item["text"],
    "fields": fields
  })

print(len(entry_list))
print(json.dumps(entry_list[0], ensure_ascii=False, indent=2))
print(json.dumps(entry_list[-1], ensure_ascii=False, indent=2))



833 833
{
  "text": "河北兵马大元帅府",
  "catalog": "宋代官制辞典/I.职官条目分类目录/第八编 军事统率机构与地方治安机构类/一、大元帅府、都督府门",
  "page": "482"
}
833
{
  "title": "河北兵马大元帅府",
  "catalog": "宋代官制辞典/I.职官条目分类目录/第八编 军事统率机构与地方治安机构类/一、大元帅府、都督府门",
  "page": "482",
  "text": "官司名。北宋靖康元年闰十一月，宋钦宗传檄，授命康王为河北兵马大元帅。十二月一日，赵构开大元帅府，以募兵勤王抗金，解救京师之围为名（《要录》卷1）。南宋建炎元年五月十日大元帅府解散（《宋会要·职官》37之2《元帅府》）。",
  "fields": {
    "简称": "①大元帅府、元帅府。《宋会要·职官》37之1：“高宗建炎元年五月二日，诏大元帅府限十日结局。”《要录》卷1靖康元年十一月己酉：“拜王（康王赵构)河北兵马大元帅。”十二月壬戌朔：“王开元帅府。”②帅府。《要录》卷1乙亥：“乃遣人伴送至帅府。”③霸府。《要录》卷1，靖康元年闰十一月己酉：“拜王河北兵马大元帅。”原注引《汪伯彦日历》：“然霸府肇启开，事出仓卒。盖靖康元年闰十一月，檄到日，康王可充兵马大元帅。”④天下兵马大元帅府。过称。《金佗粹编》卷4《行实编年》：“（靖康元年）冬，高宗皇帝以天下兵马大元帅开府河朔。”《要录》卷4，建炎元年四月癸亥：“（赵子崧）望大王遵故事，以天下兵马大元帅承制号召四方。”"
  }
}
{
  "title": "作院",
  "catalog": "宋代官制辞典/I.职官条目分类目录/第十编 地方官类之二——府州县官/五、县镇官与监当官门",
  "page": "615",
  "text": "官司名。隶本州、府、军、监。非要会州、府、军、监，皆置作院专制造兵器，每年有一定课额。作院工匠以厢兵充。设监官掌领（《长编》卷17己未、卷351癸巳，《宋平江城坊考》卷3《东南隅·作院》）。",
  "fields": {}
}


In [5]:
import json
entry_indexes = []
for entry in entry_list:
  entry_indexes.append([entry["title"], entry["page"]])

with open("./entry_indexes.json", "w", encoding="utf-8") as f:
  json.dump(entry_indexes, f, ensure_ascii=False, indent=2)

In [3]:
import sqlite3
import os
import json

db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
conn = sqlite3.connect(db_path)

# 创建表格（如果不存在）
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS chapter8t10 (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    catalog TEXT NOT NULL,
    page TEXT NOT NULL,
    text TEXT NOT NULL,
    fields TEXT
)
""")

# 将 entry_list 写入表格
cursor.executemany("""
INSERT INTO chapter8t10 (title, catalog, page, text, fields)
VALUES (?, ?, ?, ?, ?)
""", [
    (
        entry["title"],
        entry["catalog"],
        entry["page"],
        entry["text"],
        json.dumps(entry["fields"], ensure_ascii=False)
    )
    for entry in entry_list
])

conn.commit()

# 验证插入结果
cursor.execute("SELECT COUNT(*) FROM chapter8t10")
count = cursor.fetchone()[0]
print(f"成功插入 {count} 条记录")

# 查看前几条记录
cursor.execute("SELECT id, title, catalog, page FROM chapter8t10 LIMIT 5")
for row in cursor.fetchall():
    print(f"ID: {row[0]}, Title: {row[1]}, Page: {row[3]}")

cursor.close()
conn.close()



成功插入 833 条记录
ID: 1, Title: 河北兵马大元帅府, Page: 482
ID: 2, Title: 河北兵马大元帅, Page: 482
ID: 3, Title: 河北兵马元帅, Page: 482
ID: 4, Title: 河北兵马副元帅, Page: 482
ID: 5, Title: 河北兵马大元帅府参议官, Page: 482


In [ ]:
"""验证条目名称是否会重名；结论是会，因此不能用作索引"""
entry_names = set()
f = True
for item in catalog_data:
  if f and item["text"] != "第八编 军事统率机构与地方治安机构类":
    continue
  else:
    f = False
  if item["text"] == "第十一编 阶官类":
    break
  if item["type"] == "name":
    if item["text"] in entry_names:
      print(item["text"])
    else:
      entry_names.add(item["text"])
print(len(entry_names))


宣抚大使
经制司
观察判官
节度推官
观察推官
录事参军事
司户参军事
司理参军事
司法参军事
824


In [ ]:
"""验证条目名称+页码是否会重名；结论是不会，可以用作索引"""
entry_indexes = set()
for entry in entry_list:
  index = f"{entry["title"]}_{entry["page"]}"
  if index in entry_indexes:
    print(index)
  else:
    entry_indexes.add(index)

print(len(entry_indexes))

833


In [18]:
output_format = r"""
{
  "thought": "对当前信息的分析及下一步规划",
  "action": "Tasks All Finished" 或 [ {"tool": "...", "parameters": {...} } ],
  "observation": "如果是自主生成的结论则填写，否则留空等待系统返回"
}
"""
print(output_format)

tool_call_format = r"""
[
  {
    "tool": "TOOL_NAME",
    "parameters": {
      "PARAM_NAME": "PARAM_VALUE"
    }
  }
]
"""
print(tool_call_format)


{
  "thought": "对当前信息的分析及下一步规划",
  "action": "Tasks All Finished" 或 [ {"tool": "...", "parameters": {...} } ],
  "observation": "如果是自主生成的结论则填写，否则留空等待系统返回"
}


[
  {
    "tool": "TOOL_NAME",
    "parameters": {
      "PARAM_NAME": "PARAM_VALUE"
    }
  }
]



In [ ]:
def build_prompt(input_entries, entry_indexes, CoT):
  output_format = r"""
{
  "thought": "对当前信息的分析及下一步规划",
  "action": "Tasks All Finished" 或 [ {"tool": "...", "parameters": {...} } ],
  "observation": "如果是自主生成的结论则填写，否则留空等待系统返回"
}
"""

  tool_call_format = r"""
[
  {
    "tool": "TOOL_NAME",
    "parameters": {
      "PARAM_NAME": "PARAM_VALUE"
    }
  }
]
"""

  prompt = f"""
# 角色
你是一个专业的中国古代官制研究专家。你的任务是从提供给你的《宋代官制辞典》原始文本中提取结构化的“机构”与“职官”信息。

# 核心数据结构
1. 《宋代官制辞典》原始数据条目
  - index 索引
  - title 条目名称
  - catalog, page 条目在辞典中的目录结构以及页码
  - text 条目的基础说明文字
  - fields 条目的其他字段文字
2. 官制数据条目
  - index 索引
  - title, catalog, page 对应于《辞典》的条目名称，目录结构以及页码信息
  - type 条目属性分类，"机构" 或 "官职"
  - class 更加细致的条目类别，如 "军职名" "官司名" "官名" 等
  - bgn_time, end_time, bgn_event, end_event 该条目覆盖的时间区域起点与终点，以及相应发生的事件
  - superiors, subordinates “机构”条目的上下级机构，是一个列表，可以包含多个上下级机构
    - 列表中每个元素是一个字符串，代表对应的机构
  - staffing “机构”条目的人员编制，是一个列表
    - 列表中每个元素是一个三元组列表，包含职位名称(str)，类别(str)以及编制数量(num)
  - department “职官”条目所隶属的机构
    - 是一个字符串，代表对应的机构
  - grade “职官”条目的官职品级

3. 涉及到的机构与职官条目索引表：
这是一个列表，包含了全部的来自《辞典》的需要关注的机构或职官条目信息，以及他们出现在《辞典》中的页码，使用 "名称-页码" 的格式给出。名称与页码的组合将作为这些条目数据的重要索引方式。
在构建官制数据时，其中的 superiors, subordinates, department, staffing 中，如果对应的机构或职位在该索引表中，则需要使用 "名称-页码" 作为其名称字符串；不在索引表中，则直接使用原本的名称；


注意信息：
1. 《宋代官制辞典》为原始数据，是输入信息的来源；
2. 官职数据条目为待补充的信息；
3. 其中的条目与《辞典》中的条目一一对应，在初始情况下包含 title, catalog, page 三条元信息，不需要额外的修改；
4. 其余信息则包含在包括自身对应的条目，以及其他可能相关的条目中，需要进行提取后填入；
5. 机构与职官两大类条目共同被包含在数据表中，但分别拥有各自对应的属性，需要区分对待；与类别不匹配的属性保留为空值；
6. 每个条目可能在多个时间点发现变化，包括设立，改名，废除等；每个条目应当以设立作为起点，废除作为终止，如果缺少了该时间点，则应该保留占位；为了加以区分，会将处于不同时间段下（不同的开始终止时间）的条目分开记录，即这些条目有相同的元数据（title, catalog, page），但有着不同的其余属性；且这些条目的起止时间前后相连；

# 采用思维链
你必须遵循 "Thought -> Action -> Observation" 的循环：
- **Observation**: 初始的输入信息，系统的反馈结果，或是自身上一轮的结论。
- **Thought**: 分析当前已有的信息，包括输入信息，先前的决策，行动以及反馈结果，调整规划后续步骤，并生成行动决策。
- **Action**: 根据行动决策，选择直接生成结果，或调用给定的工具接口获取反馈信息。
重复以上过程，直到达成最终的目标，或是需要等待工具反馈结果

# 工具集
## 从数据表中获取数据的接口
1. 查询《辞典》数据：
  * 工具名称：search_dictionary
  * 调用方式：search_dictionary(title, page) 通过条目名称和所在辞典页码唯一确定条目
  * 工具说明：每轮工作本身会提供若干条《辞典》中的条目，工作目标即将这些条目中的信息结构化后，填入到对应的官制条目中；在《辞典》中的信息引用了其他《辞典》中的条目信息，且恰好未能在输入中提供时，可以通过该工具获取对应条目的信息；
2. 查询已有官制条目数据：
  * 工具名称：check_existing_entry
  * 调用方式：check_existing_entry(title, page) 通过条目名称和所在辞典页码唯一确定条目
  * 工具说明：完成从已有信息中提取结构化信息之后，在写入到官制数据表前，需要先确认现有的相应条目信息，进而决策新增，修改，删除等操作；查询会返回一个数据列表；若该条目包含多个时间段上的条目信息，则查询会返回所有；

## 向数据表写入数据的接口
通过由 title, page, bgn_time, end_time 四个维度组成的 metadata 唯一确定官制表中的条目，并进行修改；
注意：所有写入操作必须基于 check_existing_entry 获取的准确元数据。在连续的多步操作中，需要注意由于先前操作导致的数据项元数据变化，在后续操作中准确填入更新后的元数据，避免操作失效；

1. 填入官制条目的指定属性（创建）：
  * 工具名称：insert
  * 调用方式：insert(title, page, bgn_time, end_time, attr_key, attr_value)
  * 工具说明：填写一个当前空着的属性
2. 更新官制条目的指定属性（覆盖）：
  * 工具名称：update
  * 调用方式：update(title, page, bgn_time, end_time, attr_key, attr_value)
  * 工具说明：更新一个当前已经有值的属性，会覆盖原值，应当谨慎使用
3. 插入官制条目的指定列表属性（添加）：
  * 工具名称：append_list
  * 调用方式：append_list(title, page, bgn_time, end_time, attr_key, attr_value, index=None)
  * 工具说明：向条目中的列表类型属性插入一个新的值；如果不指定 index 则默认插入到列表末尾；
4. 修改官制条目的指定列表属性（修改）：
  * 工具名称：update_list
  * 调用方式：update_list(title, page, bgn_time, end_time, attr_key, attr_value, index)
  * 工具说明：修改条目中的列表类型属性中的指定值；会覆盖原值，应当谨慎使用；
5. 删除官制条目的指定列表属性（删除）：
  * 工具名称：remove_list
  * 调用方式：remove_list(title, page, bgn_time, end_time, attr_key, attr_value, index)
  * 工具说明：删除条目中的列表类型属性中的指定值，应当谨慎使用；其中 attr_value 为保险，要求与原值一致，防止误删；
6. 通过添加时间点的方式，拆分官制条目：
  * 工具名称：new_time_point
  * 调用方式：new_time_point(title, page, bgn_time, end_time, time_point, event)
  * 工具说明：当发现了官制条目的新的重要时间点时，需要将其扩展，额外拆分出一个条目，分别覆盖每一段时间段；确保新时间点出现在当前 metadata 选定的条目所覆盖的时间范围之内（在时间起止点之间）；该工具会自动将该条目拆分为两个新的条目，旧条目将被移除；拆分后，原本在该条目下的属性会自动复制到拆分后的两个新条目中（由底层工具保证），你只需要针对发生变化的那个时间段进行后续更新；


# 输出格式
使用 CoT 思维链的格式进行输出，采用 JSON 格式组织信息，请仅输出当前轮次的逻辑推演结果，格式如下：

{output_format}

## Thought
要求以纯字符串的形式输出思考过程，包含对上一轮结果的分析，对之后步骤的规划等内容；

## Action
如果本轮次的行动不需要调用工具等待结果，则应当以字符串形式描述本轮行动的具体要求；

如果本轮次的行动需要调用工具，则以以下方式输出结构化的工具调用指令，为一个 JSON 列表：

{tool_call_format}

如果完整的任务已经完成，在 Action 中输出字符串 "Tasks All Finished" 并结束整个流程；

## Observation
  * 若 Action 为工具调用：停止输出，等待系统返回 Observation；Action 允许一次调用多个工具，你应当尽可能将相对独立的工具调用一次性调用，如查询多个条目的信息等。
  * 若 Action 为逻辑结论或中间步骤（如：决定下一步要去搜什么）：Observation 部分应总结该步骤的发现。
  * 若 Action 为 Tasks All Finished：不输出 Observation，直接结束。”

# 当前上下文

## 核心目标
根据给定的《辞典》条目信息，提取出其中的官制条目信息，并填写到相应的数据表中；
如果给定的《辞典》条目引用了另一条《辞典》条目，影响到当前条目的信息提取，且没有出现在输入中，则应当通过工具获取该条目，补充输入数据；
如果给定的《辞典》文本中提到另一条目范围的信息，在确认该条目存在的前提下，应同时产生针对该派生条目数据的更新指令。

## 输入1：《辞典》中涉及到的全部 833 条官制索引 (title, page)
{"\n".join(entry_indexes)}

## 输入2：来自《辞典》的 X 条数据
{"\n\n".join(input_entries)}

## 上下文：已有的思维链过程
{CoT}

请你严格按照规定格式，补充完善思维链，达到最终的目标；
直接返回 JSON 格式的思维链结果，不要包含其他信息。
"""
  
  return prompt



In [24]:
import sqlite3

with open(r"D:\git-projects\vis-context-agent\song-bureaucracy\data\refined_tables\第八至十编-官制索引.json", "r", encoding="utf-8") as f:
  entry_indexes = json.load(f)

entry_indexes_texts = [f"{x[0]}-{x[1]}" for x in entry_indexes]

db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute("SELECT * FROM chapter8t10 LIMIT 5")
"""
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  title TEXT NOT NULL,
  catalog TEXT NOT NULL,
  page TEXT NOT NULL,
  text TEXT NOT NULL,
  fields TEXT
"""

input_entries = []
for row in cursor.fetchall():
  fields = json.loads(row[5])
  fields_texts = []
  for key, value in fields.items():
    fields_texts.append(f"{key}: {value}")
  entry_text = f"""
{row[1]}
文本来源：{row[2]} {row[3]}页
基本介绍：{row[4]}
{"\n".join(fields_texts)}
"""
  input_entries.append(entry_text)

# print(input_entries[0])

prompt = build_prompt(input_entries, entry_indexes_texts, "")
print(prompt)



# 角色
你是一个专业的中国古代官制研究专家。你的任务是从提供给你的《宋代官制辞典》原始文本中提取结构化的“机构”与“职官”信息。

# 核心数据结构
1. 《宋代官制辞典》原始数据条目
  - index 索引
  - title 条目名称
  - catalog, page 条目在辞典中的目录结构以及页码
  - text 条目的基础说明文字
  - fields 条目的其他字段文字
2. 官制数据条目
  - index 索引
  - title, catalog, page 对应于《辞典》的条目名称，目录结构以及页码信息
  - type 条目属性分类，"机构" 或 "官职"
  - class 更加细致的条目类别，如 "军职名" "官司名" "官名" 等
  - bgn_time, end_time, bgn_event, end_event 该条目覆盖的时间区域起点与终点，以及相应发生的事件
  - superiors, subordinates “机构”条目的上下级机构，是一个列表，可以包含多个上下级机构
  - staffing “机构”条目的人员编制，是一个列表，包含职位名称，类别以及编制数量
  - department “职官”条目所隶属的机构
  - grade “职官”条目的官职品级

注意信息：
1. 《宋代官制辞典》为原始数据，是输入信息的来源；
2. 官职数据条目为待补充的信息；
3. 其中的条目与《辞典》中的条目一一对应，在初始情况下包含 title, catalog, page 三条元信息，不需要额外的修改；
4. 其余信息则包含在包括自身对应的条目，以及其他可能相关的条目中，需要进行提取后填入；
5. 机构与职官两大类条目共同被包含在数据表中，但分别拥有各自对应的属性，需要区分对待；与类别不匹配的属性保留为空值；
6. 每个条目可能在多个时间点发现变化，包括设立，改名，废除等；每个条目应当以设立作为起点，废除作为终止，如果缺少了该时间点，则应该保留占位；为了加以区分，会将处于不同时间段下（不同的开始终止时间）的条目分开记录，即这些条目有相同的元数据（title, catalog, page），但有着不同的其余属性；且这些条目的起止时间前后相连；

# 采用思维链
你必须遵循 "Thought -> Action -> Obse

Deepseek 的返回值 2026.01.23
```
{
  "thought": "输入提供了5条《宋代官制辞典》条目信息，都位于第482页，属于“第八编 军事统率机构与地方治安机构类/一、大元帅府、都督府门”。需要将这些条目信息结构化，填入官制数据表。首先分析每个条目的类型：\n1. 河北兵马大元帅府：这是一个“官司名”，属于机构。\n2. 河北兵马大元帅：这是一个“军职名”，属于官职。\n3. 河北兵马元帅：这是一个“军职名”，属于官职。\n4. 河北兵马副元帅：这是一个“军职名”，属于官职。\n5. 河北兵马大元帅府参议官：这是一个“军职名”，属于官职。\n\n核心任务：为每个条目创建官制数据条目，并填充属性。需要先检查现有数据表（check_existing_entry）确认是否存在，再进行操作。根据文本信息，这些条目都有明确的时间范围：靖康元年（1126年）闰十一月或十二月设立，建炎元年（1127年）五月十日罢（结束）。对于“河北兵马元帅”和“河北兵马副元帅”，文本提及“靖康间”设立，但未明确具体设立日期。为简化，可以将其与“河北兵马大元帅”的起止时间保持一致，因为都隶属于大元帅府。如果后续发现不同时间点，可以用new_time_point拆分。\n\n属性填充：\n- 机构条目（河北兵马大元帅府）：需要填充type（机构）、class（官司名）、bgn_time（靖康元年闰十一月或十二月一日）、end_time（建炎元年五月十日）、bgn_event（设立）、end_event（解散/罢）、superiors（上级机构，可能需要从上下文推断，但此处无明确上级，可能直接隶属皇帝或中央？暂时留空）、subordinates（下级机构，如包含元帅、副元帅等官职？但下属官职不属于机构列表，staffing才是人员编制）、staffing（编制，需从文本提取：有大元帅、元帅、副元帅、参议官等职位）。但staffing是机构的人员编制列表，包含职位名称、类别、编制数量。文本中未明确数量，但提到“募兵五军万人之众”是军队规模，不是官职编制。可能需要从其他条目推断。暂时先填充已知职位。\n- 官职条目：需要填充type（官职）、class（军职名）、bgn_time、end_time、bgn_event、end_event、department（隶属机构，应为“河北兵马大元帅府”）、grade（品级，文本未提供，留空）。\n\n策略：先查询现有条目，然后分别插入或更新。由于这些条目可能尚未存在，我将使用insert创建新条目。为每个条目确定唯一的metadata（title, page, bgn_time, end_time）。时间点需要统一格式：采用年号纪年，转换为数字年份？但系统可能接受文本。为准确，使用文本中的年号。bgn_time: “靖康元年闰十一月”（或十二月一日），end_time: “建炎元年五月十日”。对于“河北兵马大元帅府”，文本明确设立于十二月一日，但大元帅任命在闰十一月。我可以使用“靖康元年闰十一月”作为起始事件。\n\n具体步骤：\n1. 调用check_existing_entry查询每个条目是否存在，以确认是否需要插入。\n2. 对于每个条目，使用insert填充基本属性（type, class, bgn_time, end_time, bgn_event, end_event）。\n3. 对于机构条目，填充staffing列表（使用append_list）。\n4. 对于官职条目，填充department。\n\n注意：条目之间有关联，如官职隶属于机构。在填充department时，需要确保机构条目已存在或同时创建。由于操作是顺序的，先创建机构条目可能更合适。\n\n此外，文本中提到了一些简称和别名，这些可能不需要在结构化数据中体现，除非有专门字段。目前数据结构没有简称字段，所以忽略。\n\n现在开始行动：首先查询五个条目的现有数据。",
  "action": [
    {
      "tool": "check_existing_entry",
      "parameters": {
        "title": "河北兵马大元帅府",
        "page": 482
      }
    },
    {
      "tool": "check_existing_entry",
      "parameters": {
        "title": "河北兵马大元帅",
        "page": 482
      }
    },
    {
      "tool": "check_existing_entry",
      "parameters": {
        "title": "河北兵马元帅",
        "page": 482
      }
    },
    {
      "tool": "check_existing_entry",
      "parameters": {
        "title": "河北兵马副元帅",
        "page": 482
      }
    },
    {
      "tool": "check_existing_entry",
      "parameters": {
        "title": "河北兵马大元帅府参议官",
        "page": 482
      }
    }
  ],
  "observation": ""
}
```

In [4]:
import sqlite3
import json

dict_db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
entry_db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_entries.db"

dict_table = "chapter8t10"
entry_table = "entries0124"

# 从《辞典》数据库中获取全部条目信息，并据此创建条目表
conn_dict = sqlite3.connect(dict_db_path)
cursor_dict = conn_dict.cursor()
cursor_dict.execute(f"SELECT title, catalog, page FROM {dict_table}")

entry_list = []
for row in cursor_dict.fetchall():
  entry_list.append({
    "title": row[0],
    "catalog": row[1],
    "page": row[2]
  })
print(len(entry_list), 833)

cursor_dict.close()
conn_dict.close()

"""
2. 官制数据条目
  - id 索引
  - title, catalog, page 对应于《辞典》的条目名称，目录结构以及页码信息
  - type 条目属性分类，"机构" 或 "官职"
  - subtype 更加细致的条目类别，如 "军职名" "官司名" "官名" 等
  - bgn_time, end_time, bgn_event, end_event 该条目覆盖的时间区域起点与终点，以及相应发生的事件
  - superiors, subordinates “机构”条目的上下级机构，是一个列表，可以包含多个上下级机构
  - staffing “机构”条目的人员编制，是一个列表，包含职位名称，类别以及编制数量
  - department “职官”条目所隶属的机构
  - grade “职官”条目的官职品级
"""

# 连接到条目数据库
conn_entry = sqlite3.connect(entry_db_path)
cursor_entry = conn_entry.cursor()

cursor_entry.execute(f"""
CREATE TABLE IF NOT EXISTS {entry_table} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    catalog TEXT NOT NULL,
    page TEXT NOT NULL,
    type TEXT,
    subtype TEXT,
    bgn_time TEXT,
    bgn_event TEXT,
    end_time TEXT,
    end_event TEXT,
    superiors TEXT,
    subordinates TEXT,
    staffing TEXT,
    department TEXT,
    grade TEXT
)
""")

# 从 entry_list 插入初始数据
cursor_entry.executemany(f"""
INSERT INTO {entry_table} (title, catalog, page, bgn_time, bgn_event, end_time, end_event, superiors, subordinates, staffing)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", [
    (
        entry["title"],
        entry["catalog"],
        entry["page"],
        "-INF",      # bgn_time 初始值
        "始置",      # bgn_event 初始值
        "INF",       # end_time 初始值
        "罢置",      # end_event 初始值
        "[]",        # superiors 初始值（空列表）
        "[]",        # subordinates 初始值（空列表）
        "[]"         # staffing 初始值（空列表）
    )
    for entry in entry_list
])

conn_entry.commit()

# 验证插入结果
cursor_entry.execute(f"SELECT COUNT(*) FROM {entry_table}")
count = cursor_entry.fetchone()[0]
print(f"成功插入 {count} 条记录到 {entry_table} 表")

# 查看前几条记录
cursor_entry.execute(f'SELECT id, title, catalog, page, bgn_time, bgn_event, end_time, end_event FROM {entry_table} LIMIT 5')
for row in cursor_entry.fetchall():
    print(f"Id: {row[0]}, Title: {row[1]}, Page: {row[3]}, bgn_time: {row[4]}, bgn_event: {row[5]}, end_time: {row[6]}, end_event: {row[7]}")

cursor_entry.close()
conn_entry.close()


833 833
成功插入 833 条记录到 entries0124 表
Id: 1, Title: 河北兵马大元帅府, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 2, Title: 河北兵马大元帅, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 3, Title: 河北兵马元帅, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 4, Title: 河北兵马副元帅, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 5, Title: 河北兵马大元帅府参议官, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置


In [ ]:
"""
构建信息获取与写入工具
传入参数为 (conn, **kwargs) 其中 conn 为对应数据库的连接对象；
返回值为 JSON 对象：
- 读操作：返回对象或对象列表
- 写操作：返回更新后的对象（或对象列表）
- 错误：返回 {"error": "..."}
"""

import json

# 表名
# - dict_table: 《辞典》条目表
# - entry_table: 官制条目表
dict_table = "chapter8t10"
entry_table = "entries0124"

# 字段白名单（防 SQL 注入）
_entry_attr_whitelist = {
  "type",
  "subtype",
  "bgn_time",
  "bgn_event",
  "end_time",
  "end_event",
  "superiors",
  "subordinates",
  "staffing",
  "department",
  "grade",
}
_entry_list_attr_whitelist = {"superiors", "subordinates", "staffing"}


def _loads_list(v):
  if v is None or v == "":
    return []
  try:
    x = json.loads(v)
  except Exception:
    return []
  return x if isinstance(x, list) else []


def _dumps_list(v):
  if v is None:
    return "[]"
  if isinstance(v, list):
    return json.dumps(v, ensure_ascii=False)
  # 允许直接传入 JSON 字符串
  if isinstance(v, str):
    try:
      x = json.loads(v)
      if isinstance(x, list):
        return json.dumps(x, ensure_ascii=False)
    except Exception:
      pass
  raise ValueError("列表字段要求 list 或可解析为 list 的 JSON 字符串")


def _row_to_entry_obj(row):
  return {
    "id": row[0],
    "title": row[1],
    "catalog": row[2],
    "page": row[3],
    "type": row[4],
    "subtype": row[5],
    "bgn_time": row[6],
    "bgn_event": row[7],
    "end_time": row[8],
    "end_event": row[9],
    "superiors": _loads_list(row[10]),
    "subordinates": _loads_list(row[11]),
    "staffing": _loads_list(row[12]),
    "department": row[13],
    "grade": row[14],
  }


def _fetch_entry_by_id(cursor, entry_id):
  cursor.execute(
    f"""
    SELECT id, title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
           superiors, subordinates, staffing, department, grade
    FROM {entry_table}
    WHERE id = ?
    """,
    (entry_id,),
  )
  row = cursor.fetchone()
  if row is None:
    return None
  return _row_to_entry_obj(row)


def _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time):
  cursor.execute(
    f"""
    SELECT id
    FROM {entry_table}
    WHERE title = ? AND page = ? AND bgn_time = ? AND end_time = ?
    """,
    (title, str(page), bgn_time, end_time),
  )
  row = cursor.fetchone()
  return None if row is None else row[0]


# 1. 查询《辞典》数据
def search_dictionary(conn, title, page):
  cursor = conn.cursor()
  try:
    cursor.execute(
      f"""
      SELECT id, title, catalog, page, text, fields
      FROM {dict_table}
      WHERE title = ? AND page = ?
      """,
      (title, str(page)),
    )
    row = cursor.fetchone()
    if row is None:
      return {"error": f"未找到条目: {title} (页码: {page})"}

    return {
      "id": row[0],
      "title": row[1],
      "catalog": row[2],
      "page": row[3],
      "text": row[4],
      "fields": json.loads(row[5]) if row[5] else {},
    }
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 2. 查询已有官制条目数据
def check_existing_entry(conn, title, page):
  cursor = conn.cursor()
  try:
    cursor.execute(
      f"""
      SELECT id, title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
             superiors, subordinates, staffing, department, grade
      FROM {entry_table}
      WHERE title = ? AND page = ?
      ORDER BY bgn_time, end_time, id
      """,
      (title, str(page)),
    )

    rows = cursor.fetchall()
    # 按文档约定：查不到返回空列表（不算错误）
    return [_row_to_entry_obj(r) for r in rows]
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 3. 填入官制条目的指定属性（创建）
def insert(conn, title, page, bgn_time, end_time, attr_key, attr_value):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_attr_whitelist:
      return {"error": f"无效的属性名: {attr_key}"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_value = cursor.fetchone()[0]
    if current_value is not None and current_value != "" and current_value != "[]":
      return {"error": f"属性 {attr_key} 已有值: {current_value}，请使用 update"}

    if attr_key in _entry_list_attr_whitelist:
      attr_value = _dumps_list(attr_value)

    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (attr_value, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "写入成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 4. 更新官制条目的指定属性（覆盖）
def update(conn, title, page, bgn_time, end_time, attr_key, attr_value):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_attr_whitelist:
      return {"error": f"无效的属性名: {attr_key}"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    if attr_key in _entry_list_attr_whitelist:
      attr_value = _dumps_list(attr_value)

    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (attr_value, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 5. 插入官制条目的指定列表属性（添加）
def append_list(conn, title, page, bgn_time, end_time, attr_key, attr_value, index=None):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_list_attr_whitelist:
      return {"error": f"属性 {attr_key} 不是列表类型"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_list = _loads_list(cursor.fetchone()[0])

    if index is None:
      current_list.append(attr_value)
    else:
      if not isinstance(index, int):
        return {"error": "index 必须为 int 或 None"}
      if index < 0 or index > len(current_list):
        return {"error": f"index 越界: {index} (len={len(current_list)})"}
      current_list.insert(index, attr_value)

    new_list_str = json.dumps(current_list, ensure_ascii=False)
    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (new_list_str, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 6. 修改官制条目的指定列表属性（修改）
def update_list(conn, title, page, bgn_time, end_time, attr_key, attr_value, index):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_list_attr_whitelist:
      return {"error": f"属性 {attr_key} 不是列表类型"}
    if not isinstance(index, int):
      return {"error": "index 必须为 int"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_list = _loads_list(cursor.fetchone()[0])

    if index < 0 or index >= len(current_list):
      return {"error": f"索引 {index} 超出范围 (列表长度: {len(current_list)})"}

    current_list[index] = attr_value

    new_list_str = json.dumps(current_list, ensure_ascii=False)
    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (new_list_str, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 7. 删除官制条目的指定列表属性（删除）
def remove_list(conn, title, page, bgn_time, end_time, attr_key, attr_value, index):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_list_attr_whitelist:
      return {"error": f"属性 {attr_key} 不是列表类型"}
    if not isinstance(index, int):
      return {"error": "index 必须为 int"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_list = _loads_list(cursor.fetchone()[0])

    if index < 0 or index >= len(current_list):
      return {"error": f"索引 {index} 超出范围 (列表长度: {len(current_list)})"}

    if current_list[index] != attr_value:
      return {
        "error": "索引位置的值不匹配，拒绝删除",
        "current": current_list[index],
        "expected": attr_value,
      }

    current_list.pop(index)

    new_list_str = json.dumps(current_list, ensure_ascii=False)
    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (new_list_str, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 8. 通过添加时间点的方式，拆分官制条目
def new_time_point(conn, title, page, bgn_time, end_time, time_point, event):
  cursor = conn.cursor()
  try:
    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"""
      SELECT id, title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
             superiors, subordinates, staffing, department, grade
      FROM {entry_table}
      WHERE id = ?
      """,
      (entry_id,),
    )
    row = cursor.fetchone()
    if row is None:
      return {"error": "读取待拆分条目失败"}

    old = _row_to_entry_obj(row)

    # 插入两条新记录（复制旧记录其他字段）
    cursor.execute(
      f"""
      INSERT INTO {entry_table}
      (title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
       superiors, subordinates, staffing, department, grade)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
      """,
      (
        old["title"],
        old["catalog"],
        str(old["page"]),
        old["type"],
        old["subtype"],
        old["bgn_time"],
        old["bgn_event"],
        time_point,
        event,
        json.dumps(old["superiors"], ensure_ascii=False),
        json.dumps(old["subordinates"], ensure_ascii=False),
        json.dumps(old["staffing"], ensure_ascii=False),
        old["department"],
        old["grade"],
      ),
    )
    id1 = cursor.lastrowid

    cursor.execute(
      f"""
      INSERT INTO {entry_table}
      (title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
       superiors, subordinates, staffing, department, grade)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
      """,
      (
        old["title"],
        old["catalog"],
        str(old["page"]),
        old["type"],
        old["subtype"],
        time_point,
        event,
        old["end_time"],
        old["end_event"],
        json.dumps(old["superiors"], ensure_ascii=False),
        json.dumps(old["subordinates"], ensure_ascii=False),
        json.dumps(old["staffing"], ensure_ascii=False),
        old["department"],
        old["grade"],
      ),
    )
    id2 = cursor.lastrowid

    # 删除旧记录
    cursor.execute(
      f"DELETE FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )

    conn.commit()

    obj1 = _fetch_entry_by_id(cursor, id1)
    obj2 = _fetch_entry_by_id(cursor, id2)
    if obj1 is None or obj2 is None:
      return {"error": "拆分成功但读取新对象失败"}
    return [obj1, obj2]
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}



In [ ]:
"""
Tools 类：封装所有工具函数，管理数据库连接，并提供引用完整性检查
"""

import sqlite3
import re


class Tools:
  def __init__(self, dict_db_path, entry_db_path, indexes):
    """
    初始化 Tools 实例
    
    Args:
      dict_db_path: 《辞典》数据库路径
      entry_db_path: 官制条目数据库路径
      indexes: 索引集合，格式为 set of "名称-页码" 字符串
    """
    self.dict_db_path = dict_db_path
    self.entry_db_path = entry_db_path
    self.indexes = indexes
    
    # 在初始化时建立数据库连接
    self._dict_conn = sqlite3.connect(self.dict_db_path)
    self._entry_conn = sqlite3.connect(self.entry_db_path)
  
  def close(self):
    """关闭所有数据库连接"""
    if self._dict_conn is not None:
      self._dict_conn.close()
      self._dict_conn = None
    if self._entry_conn is not None:
      self._entry_conn.close()
      self._entry_conn = None
  
  def __enter__(self):
    return self
  
  def __exit__(self, exc_type, exc_val, exc_tb):
    self.close()
  
  def _is_indexed_format(self, name):
    """
    检查字符串是否符合 "名称-页码" 格式
    
    Args:
      name: 待检查的字符串
    
    Returns:
      bool: 如果符合格式返回 True，否则返回 False
    """
    if not isinstance(name, str) or not name:
      return False
    # 检查是否以 "-数字" 结尾
    return bool(re.search(r'-\d+$', name))
  
  def _validate_entry_ref(self, name):
    """
    验证条目引用是否存在于索引中
    仅当 name 符合 "名称-页码" 格式时才检查
    
    Args:
      name: 条目名称字符串
    
    Returns:
      dict: 如果验证失败返回 {"error": "..."}, 否则返回 None
    """
    # 如果不是索引格式，跳过检查
    if not self._is_indexed_format(name):
      return None
    
    # 检查是否在索引中
    if name not in self.indexes:
      return {"error": f"引用的条目不存在于索引中: {name}"}
    return None
  
  def _validate_string_list(self, items, field_name):
    """
    验证字符串列表中的所有条目引用
    
    Args:
      items: 字符串列表
      field_name: 字段名（用于错误提示）
    
    Returns:
      dict: 如果验证失败返回 {"error": "..."}, 否则返回 None
    """
    if not isinstance(items, list):
      return {"error": f"字段 {field_name} 必须为 list 类型"}
    
    for i, item in enumerate(items):
      if not isinstance(item, str):
        return {"error": f"字段 {field_name} 的元素 [{i}] 必须为 str 类型"}
      
      # 只检查符合索引格式的字符串
      err = self._validate_entry_ref(item)
      if err is not None:
        return {"error": f"字段 {field_name} 的元素 [{i}] 引用无效: {err['error']}"}
    
    return None
  
  def _validate_staffing_list(self, items):
    """
    验证 staffing 列表中的所有条目引用
    staffing 格式：[[职位名称(str), 类别(str), 编制数量(num)], ...]
    
    Args:
      items: staffing 列表
    
    Returns:
      dict: 如果验证失败返回 {"error": "..."}, 否则返回 None
    """
    if not isinstance(items, list):
      return {"error": "字段 staffing 必须为 list 类型"}
    
    for i, item in enumerate(items):
      if not isinstance(item, list):
        return {"error": f"字段 staffing 的元素 [{i}] 必须为 list 类型（三元组）"}
      if len(item) != 3:
        return {"error": f"字段 staffing 的元素 [{i}] 必须包含 3 个元素（职位名称, 类别, 编制数量）"}
      
      position_name = item[0]
      if not isinstance(position_name, str):
        return {"error": f"字段 staffing 的元素 [{i}] 的职位名称必须为 str 类型"}
      
      # 只检查符合索引格式的职位名称
      err = self._validate_entry_ref(position_name)
      if err is not None:
        return {"error": f"字段 staffing 的元素 [{i}] 职位名称引用无效: {err['error']}"}
    
    return None
  
  # ========== 读取工具 ==========
  
  def search_dictionary(self, title, page):
    """查询《辞典》数据"""
    return search_dictionary(self._dict_conn, title, page)
  
  def check_existing_entry(self, title, page):
    """查询已有官制条目数据"""
    return check_existing_entry(self._entry_conn, title, page)
  
  # ========== 写入工具 ==========
  
  def insert(self, title, page, bgn_time, end_time, attr_key, attr_value):
    """填入官制条目的指定属性（创建）"""
    # 对于引用字段，需要额外验证
    if attr_key == "department":
      if attr_value is not None and attr_value != "":
        # department 是字符串
        err = self._validate_entry_ref(attr_value)
        if err is not None:
          return err
    
    elif attr_key in ["superiors", "subordinates"]:
      # 字符串列表
      err = self._validate_string_list(attr_value, attr_key)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      # staffing 是三元组列表
      err = self._validate_staffing_list(attr_value)
      if err is not None:
        return err
    
    return insert(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value)
  
  def update(self, title, page, bgn_time, end_time, attr_key, attr_value):
    """更新官制条目的指定属性（覆盖）"""
    # 对于引用字段，需要额外验证
    if attr_key == "department":
      if attr_value is not None and attr_value != "":
        err = self._validate_entry_ref(attr_value)
        if err is not None:
          return err
    
    elif attr_key in ["superiors", "subordinates"]:
      err = self._validate_string_list(attr_value, attr_key)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      err = self._validate_staffing_list(attr_value)
      if err is not None:
        return err
    
    return update(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value)
  
  def append_list(self, title, page, bgn_time, end_time, attr_key, attr_value, index=None):
    """插入官制条目的指定列表属性（添加）"""
    # 对于引用列表字段，需要验证新增的元素
    if attr_key in ["superiors", "subordinates"]:
      if not isinstance(attr_value, str):
        return {"error": f"字段 {attr_key} 的元素必须为 str 类型"}
      err = self._validate_entry_ref(attr_value)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      if not isinstance(attr_value, list) or len(attr_value) != 3:
        return {"error": "staffing 的元素必须为三元组 list [职位名称, 类别, 编制数量]"}
      if not isinstance(attr_value[0], str):
        return {"error": "staffing 元素的职位名称必须为 str 类型"}
      err = self._validate_entry_ref(attr_value[0])
      if err is not None:
        return err
    
    return append_list(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value, index)
  
  def update_list(self, title, page, bgn_time, end_time, attr_key, attr_value, index):
    """修改官制条目的指定列表属性（修改）"""
    # 对于引用列表字段，需要验证更新后的元素
    if attr_key in ["superiors", "subordinates"]:
      if not isinstance(attr_value, str):
        return {"error": f"字段 {attr_key} 的元素必须为 str 类型"}
      err = self._validate_entry_ref(attr_value)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      if not isinstance(attr_value, list) or len(attr_value) != 3:
        return {"error": "staffing 的元素必须为三元组 list [职位名称, 类别, 编制数量]"}
      if not isinstance(attr_value[0], str):
        return {"error": "staffing 元素的职位名称必须为 str 类型"}
      err = self._validate_entry_ref(attr_value[0])
      if err is not None:
        return err
    
    return update_list(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value, index)
  
  def remove_list(self, title, page, bgn_time, end_time, attr_key, attr_value, index):
    """删除官制条目的指定列表属性（删除）"""
    # remove_list 不需要额外的引用验证，因为是删除操作
    return remove_list(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value, index)
  
  def new_time_point(self, title, page, bgn_time, end_time, time_point, event):
    """通过添加时间点的方式，拆分官制条目"""
    return new_time_point(self._entry_conn, title, page, bgn_time, end_time, time_point, event)


In [ ]:
# 示例：使用 Tools 类

# 1. 初始化 Tools 实例
dict_db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
entry_db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_entries.db"

# 加载索引并转换为 "名称-页码" 格式的集合
import json
with open(r"D:\git-projects\vis-context-agent\song-bureaucracy\data\refined_tables\第八至十编-官制索引.json", "r", encoding="utf-8") as f:
  entry_indexes_list = json.load(f)

# 将 [[title, page], ...] 格式转换为 set of "名称-页码"
entry_indexes = {f"{item[0]}-{item[1]}" for item in entry_indexes_list}
print(f"加载了 {len(entry_indexes)} 条索引")

# 创建 Tools 实例
tools = Tools(dict_db_path, entry_db_path, indexes=entry_indexes)

# 2. 查询《辞典》数据
result = tools.search_dictionary("河北兵马大元帅府", 482)
print("\n查询《辞典》数据:")
print(json.dumps(result, ensure_ascii=False, indent=2))

# 3. 查询官制条目数据
result = tools.check_existing_entry("河北兵马大元帅府", 482)
print("\n查询官制条目数据:")
if isinstance(result, list):
  print(f"找到 {len(result)} 条记录")
  if result:
    print(json.dumps(result[0], ensure_ascii=False, indent=2))
else:
  print(json.dumps(result, ensure_ascii=False, indent=2))

# 4. 测试写入操作（insert）
result = tools.insert("河北兵马大元帅府", 482, "-INF", "INF", "type", "机构")
print("\n插入 type 字段:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - type={result['type']}")

# 5. 测试 department 引用验证（有效引用）
result = tools.insert("河北兵马大元帅", 482, "-INF", "INF", "department", "河北兵马大元帅府-482")
print("\n插入 department 字段（有效引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - department={result['department']}")

# 6. 测试 department 引用验证（无效引用）
result = tools.insert("河北兵马元帅", 482, "-INF", "INF", "department", "不存在的机构-999")
print("\n插入 department 字段（无效引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - department={result['department']}")

# 7. 测试 department 引用验证（非索引格式，跳过检查）
result = tools.insert("河北兵马副元帅", 482, "-INF", "INF", "department", "大元帅府")
print("\n插入 department 字段（非索引格式，跳过检查）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - department={result['department']}")

# 8. 测试 superiors 字段（字符串列表）
result = tools.insert(
  "河北兵马大元帅府", 482, "-INF", "INF", 
  "subordinates", 
  [
    "河北兵马大元帅-482",
    "河北兵马元帅-482",
    "河北兵马副元帅-482"
  ]
)
print("\n插入 subordinates 字段（字符串列表，有效引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - subordinates 包含 {len(result['subordinates'])} 个条目")

# 9. 测试 staffing 字段（三元组列表）
result = tools.insert(
  "河北兵马大元帅府", 482, "-INF", "INF",
  "staffing",
  [
    ["河北兵马大元帅-482", "军职", 1],
    ["河北兵马元帅-482", "军职", 1],
    ["参议官", "幕职官", 2]  # 非索引格式，跳过检查
  ]
)
print("\n插入 staffing 字段（三元组列表，混合引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - staffing 包含 {len(result['staffing'])} 个职位")

# 10. 关闭连接
tools.close()
print("\n已关闭所有数据库连接")
